<table align="left">
  <td><a target="_blank" href="https://colab.research.google.com/github/marcoteran/ml/blob/master/notebooks/02_ml_boosting_fraude.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></td>
  <td><a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marcoteran/ml/blob/master/notebooks/02_ml_boosting_fraude.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle"/></a></td>
</table>
<br><br>

# Árboles, ensembles, boosting y Optuna sobre el mismo pipeline

**Curso:** Aprendizaje Automático — SI7009 - 1 (5553)
**Sesión 1 · Parte 2:** Árboles, ensembles, boosting moderno y optimización bayesiana
**Universidad:** EAFIT
**Profesor:** Andrés Vásquez Restrepo
**Dataset:** Credit Card Transactions Fraud Detection (Kaggle `kartik2112/fraud-detection`)

---

**Pregunta de la sesión:** con los mismos datos, las mismas features, el mismo preprocesamiento y los **mismos folds** del notebook 01, ¿qué modelo supera de verdad a la regresión logística, y se sostiene en datos futuros?

Solo cambia el `model` al final del pipeline: `build_pipeline(model=...)`.

1. [Cómo ejecutar](#1) · 2. [Configuración](#2) · 3. [Datos y pipeline de la parte 1](#3) · 4. [Baseline a superar](#4) · 5. [Árboles y bias–variance](#5) · 6. [Ensembles](#6) · 7. [Boosting moderno](#7) · 8. [Balanceo](#8) · 9. [Optuna](#9) · 10. [Modelo final e inferencia](#10) · 11. [Ejercicios](#11)

<a name="1"></a>
## 1. Cómo ejecutar

**Local con `uv`**, desde la raíz del repositorio: `uv sync` y luego `uv run jupyter lab notebooks/02_ml_boosting_fraude.ipynb` (o el kernel `.venv` en VS Code).

**Colab / Kaggle:** ejecutar la celda siguiente, que instala `xgboost`, `lightgbm`, `catboost`, `optuna` e `imbalanced-learn`.

**Artefactos del notebook 01:** si existen (`notebooks/artifacts/` local, `MyDrive/SI7009_ML/artifacts` en Colab con Drive, `/kaggle/working/artifacts`), se usan como barra a superar. Si no, el baseline se **reconstruye** con los mismos bloques compartidos: da exactamente el mismo resultado.

In [ ]:
# =============================================================================
# 1.1 Environment detection and cloud install
# =============================================================================
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn>=1.5", "imbalanced-learn",
                    "xgboost", "lightgbm", "catboost", "optuna", "kagglehub[pandas-datasets]>=1.0.2", "joblib"],
                   check=True)
    print("Cloud environment: dependencies installed.")
else:
    print("Local environment: dependencies come from pyproject.toml / uv.lock (run `uv sync`).")

<a name="2"></a>
## 2. Configuración

Mismas constantes que el notebook 01 (`RANDOM_STATE`, muestra, costos): es lo que hace comparables los resultados.

In [ ]:
# =============================================================================
# 2.1 Imports, configuration and helpers (same constants as notebook 01)
# =============================================================================
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score, make_scorer,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold, TunedThresholdClassifierCV,
                                     cross_validate)
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FAST_DEMO_MODE = True
FAST_SAMPLE_SIZE = 150_000
TARGET = "is_fraud"
C_FN, C_FP = 500, 10
N_SPLITS = 5
N_TRIALS = 20 if FAST_DEMO_MODE else 60

USE_GOOGLE_DRIVE = True        # only used in Colab
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/SI7009_ML/artifacts")
elif IN_KAGGLE:
    ARTIFACT_DIR = Path("/kaggle/working/artifacts")
else:
    ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def expected_cost(y_true, y_pred) -> float:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return C_FN * fn + C_FP * fp


def binary_report(y_true, proba, threshold: float = 0.5, name: str = "") -> dict:
    y_pred = (np.asarray(proba) >= threshold).astype(int)
    return {"model": name, "threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "pr_auc": average_precision_score(y_true, proba), "roc_auc": roc_auc_score(y_true, proba),
            "alerts": int(y_pred.sum()), "expected_cost": expected_cost(y_true, y_pred)}


print(f"scikit-learn {sklearn.__version__} | lightgbm {lgb.__version__} | optuna {optuna.__version__} | artifacts: {ARTIFACT_DIR.resolve()}")

<a name="3"></a>
## 3. Datos y pipeline de la parte 1

Las tres celdas `SHARED PREP BLOCK` son **copia exacta** de las del notebook 01: misma muestra, mismo split, mismas features y mismo preprocesamiento. Además se fijan los mismos folds `skf`.

In [ ]:
# =============================================================================
# 4.1 Locate (or download) the data
# =============================================================================
DATASET_SLUG = "kartik2112/fraud-detection"
FILES = {"history": "fraudTrain.csv", "production": "fraudTest.csv"}
CANDIDATE_DIRS = [Path("data"), Path("../data"), Path("/content/data"), Path("/kaggle/input/fraud-detection")]


def find_or_download_fraud_data() -> Path:
    for folder in CANDIDATE_DIRS:
        if all((folder / name).exists() for name in FILES.values()):
            return folder
    import kagglehub  # public dataset
    return Path(kagglehub.dataset_download(DATASET_SLUG))


DATA_DIR = find_or_download_fraud_data()
print("Data folder:", DATA_DIR)

In [ ]:
# =============================================================================
# 4.2 SHARED PREP BLOCK (1/3, copiar igual en 02): load, clean, split
# =============================================================================
# Requires RANDOM_STATE, TARGET, FAST_DEMO_MODE, FAST_SAMPLE_SIZE (cell 3.1) and DATA_DIR, FILES (cell 4.1).
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_COLUMNS = [                      # the raw schema production must send
    "trans_date_trans_time", "category", "amt", "gender", "state",
    "lat", "long", "city_pop", "dob", "merch_lat", "merch_long",
]


def load_fraud_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, usecols=INPUT_COLUMNS + [TARGET])


def load_history(fast: bool = FAST_DEMO_MODE) -> pd.DataFrame:
    df = load_fraud_csv(DATA_DIR / FILES["history"])
    if fast and len(df) > FAST_SAMPLE_SIZE:
        df, _ = train_test_split(df, train_size=FAST_SAMPLE_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE)
    return df.reset_index(drop=True)


history = load_history()
X_train, X_test, y_train, y_test = train_test_split(
    history[INPUT_COLUMNS], history[TARGET], test_size=0.20, stratify=history[TARGET], random_state=RANDOM_STATE
)
print(f"history: {history.shape} | fraud rate {history[TARGET].mean():.3%}")
print(f"train: {X_train.shape} ({y_train.sum()} frauds) | test: {X_test.shape} ({y_test.sum()} frauds)")
X_train.head(3)

In [ ]:
# =============================================================================
# 4.3 SHARED PREP BLOCK (2/3, copiar igual en 02): raw schema and reusable transformers
# =============================================================================
# Self-contained: this cell must be defined before joblib.load() of the artifact.
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

INPUT_SCHEMA = {
    "trans_date_trans_time": "datetime string",
    "category": "string",
    "amt": "float",
    "gender": "string",
    "state": "string",
    "lat": "float",
    "long": "float",
    "city_pop": "int",
    "dob": "date string",
    "merch_lat": "float",
    "merch_long": "float",
}
NUMERIC_FEATURES = ["amt", "log_amt", "log_city_pop", "age", "distance_km"]
CATEGORICAL_FEATURES = ["category", "gender", "state", "hour", "day_of_week"]


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


class FraudFeatureBuilder(BaseEstimator, TransformerMixin):
    """Raw transaction columns -> model features. Stateless: learns nothing in fit."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        ts = pd.to_datetime(X["trans_date_trans_time"], errors="coerce")
        dob = pd.to_datetime(X["dob"], errors="coerce")
        amt = pd.to_numeric(X["amt"], errors="coerce")
        return pd.DataFrame({
            "amt": amt,
            "log_amt": np.log1p(amt.clip(lower=0)),
            "log_city_pop": np.log1p(pd.to_numeric(X["city_pop"], errors="coerce")),
            "age": (ts - dob).dt.days / 365.25,
            "distance_km": haversine_km(X["lat"], X["long"], X["merch_lat"], X["merch_long"]),
            "category": X["category"].astype("object"),
            "gender": X["gender"].astype("object"),
            "state": X["state"].astype("object"),
            "hour": ts.dt.hour.astype("float").astype("Int64").astype("object"),
            "day_of_week": ts.dt.dayofweek.astype("float").astype("Int64").astype("object"),
        }, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.array(NUMERIC_FEATURES + CATEGORICAL_FEATURES, dtype=object)


class QuantileClipper(BaseEstimator, TransformerMixin):
    """Clip each column to [lower, upper] quantiles learned on the training data."""

    def __init__(self, lower: float | None = None, upper: float | None = 0.999):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(self.lower) if self.lower is not None else None
        self.upper_ = X.quantile(self.upper) if self.upper is not None else None
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X.clip(lower=self.lower_, upper=self.upper_, axis=1)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_


def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Check the raw input before scoring: required columns present, extra columns dropped."""
    missing = [c for c in INPUT_SCHEMA if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")
    extra = [c for c in df.columns if c not in INPUT_SCHEMA]
    if extra:
        print(f"[validate_schema] ignoring extra columns: {extra}")
    return df[list(INPUT_SCHEMA)]

In [ ]:
# =============================================================================
# 4.4 SHARED PREP BLOCK (3/3, copiar igual en 02): preprocessing + pipeline builders
# =============================================================================
# In 02 only the `model` argument changes: same features, same preprocessing.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor(clip_upper: float | None = 0.999, min_frequency: int = 20) -> Pipeline:
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clip", QuantileClipper(upper=clip_upper)),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency, sparse_output=False)),
    ])
    columns = ColumnTransformer([("num", numeric, NUMERIC_FEATURES), ("cat", categorical, CATEGORICAL_FEATURES)],
                                verbose_feature_names_out=False)
    return Pipeline([("features", FraudFeatureBuilder()), ("columns", columns)])


def build_pipeline(model=None, **prep_kwargs) -> Pipeline:
    model = model if model is not None else LogisticRegression(max_iter=2000)
    return Pipeline([("prep", build_preprocessor(**prep_kwargs)), ("model", model)]).set_output(transform="pandas")

In [ ]:
# =============================================================================
# 4.5 Same folds as notebook 01, and one CV helper for every model
# =============================================================================
skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
SCALE_POS_WEIGHT = float((y_train == 0).sum() / (y_train == 1).sum())
cv_results: list[dict] = []


def cv_eval(name: str, estimator, cv=skf, n_jobs: int = -1) -> dict:
    """Cross-validate on the shared folds and store PR-AUC, ROC-AUC, Brier and time."""
    t0 = time.perf_counter()
    res = cross_validate(estimator, X_train, y_train, cv=cv, n_jobs=n_jobs, return_train_score=True,
                         scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"})
    row = {"model": name, "pr_auc": res["test_pr_auc"].mean(), "pr_auc_std": res["test_pr_auc"].std(),
           "train_pr_auc": res["train_pr_auc"].mean(), "roc_auc": res["test_roc_auc"].mean(),
           "brier": -res["test_brier"].mean(), "seconds": time.perf_counter() - t0}
    cv_results.append(row)
    return row


def leaderboard() -> pd.DataFrame:
    return pd.DataFrame(cv_results).drop_duplicates("model", keep="last").sort_values("pr_auc", ascending=False)


print(f"train {X_train.shape} | frauds {y_train.sum()} | scale_pos_weight = {SCALE_POS_WEIGHT:.1f}")

<a name="4"></a>
## 4. El baseline a superar

La regresión logística ajustada en el notebook 01. Si su `model_card.json` está disponible se usan sus parámetros; si no, se repite la misma búsqueda del notebook 01 (misma semilla, mismos candidatos), que devuelve los mismos parámetros.

In [ ]:
# =============================================================================
# 4.1 Load (or rebuild) the logistic-regression baseline from notebook 01
# =============================================================================
card_path = ARTIFACT_DIR / "model_card.json"
if card_path.exists():
    baseline_card = json.loads(card_path.read_text())
    baseline_params = baseline_card["params"]
    print("baseline params loaded from notebook 01:", baseline_params)
else:
    print("model_card.json not found: rebuilding notebook 01's search (same seed, same candidates)...")
    search_cv = StratifiedKFold(3 if FAST_DEMO_MODE else N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(build_pipeline(), {
        "model__C": [0.01, 0.1, 1.0], "model__class_weight": [None, "balanced"],
        "prep__columns__num__clip__upper": [0.99, 0.999]}, scoring="average_precision", cv=search_cv, n_jobs=-1)
    random = RandomizedSearchCV(build_pipeline(), {
        "model__C": loguniform(1e-3, 1e2), "model__class_weight": [None, "balanced"],
        "prep__columns__num__clip__upper": [0.95, 0.99, 0.995, 0.999, None]},
        n_iter=12, scoring="average_precision", cv=search_cv, n_jobs=-1, random_state=RANDOM_STATE)
    baseline_params = max([grid.fit(X_train, y_train), random.fit(X_train, y_train)], key=lambda s: s.best_score_).best_params_
    print("rebuilt baseline params:", baseline_params)

baseline_pipe = build_pipeline().set_params(**baseline_params)
cv_eval("logreg (baseline 01)", baseline_pipe)
leaderboard()

<a name="5"></a>
## 5. Árboles de decisión y bias–variance

Un árbol se hace más flexible con la profundidad. Comparando el score en **train** y en **validación** se ve el trade-off:

- árbol poco profundo: los dos scores son bajos y parecidos (**bias** alto);
- árbol muy profundo: train casi perfecto y validación que cae (**variance** alta).

In [ ]:
# =============================================================================
# 5.1 Depth sweep: train vs validation PR-AUC
# =============================================================================
depth_rows = []
for depth in [2, 4, 6, 8, 12, 16, None]:
    tree = build_pipeline(DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE))
    res = cross_validate(tree, X_train, y_train, cv=skf, scoring="average_precision", return_train_score=True, n_jobs=-1)
    depth_rows.append({"max_depth": "None" if depth is None else depth,
                       "train_pr_auc": res["train_score"].mean(), "val_pr_auc": res["test_score"].mean()})
depth_table = pd.DataFrame(depth_rows)

ax = depth_table.plot(x="max_depth", y=["train_pr_auc", "val_pr_auc"], marker="o", figsize=(8, 4.5))
ax.set(title="Profundidad del árbol: train frente a validación", xlabel="max_depth", ylabel="PR-AUC")
plt.show()
depth_table

In [ ]:
# =============================================================================
# 5.2 Post-pruning by cost-complexity (ccp_alpha chosen with CV)
# =============================================================================
Z_train = build_preprocessor().fit_transform(X_train)            # only to compute the pruning path
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(Z_train, y_train)
alphas = np.unique(np.quantile(path.ccp_alphas[path.ccp_alphas > 0], np.linspace(0.5, 0.99, 8)))

pruning = GridSearchCV(build_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
                       {"model__ccp_alpha": alphas}, scoring="average_precision", cv=skf, n_jobs=-1).fit(X_train, y_train)
best_alpha = pruning.best_params_["model__ccp_alpha"]
print(f"best ccp_alpha = {best_alpha:.2e} | CV PR-AUC = {pruning.best_score_:.3f}")

cv_eval("tree depth=6", build_pipeline(DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)))
cv_eval("tree pruned (ccp_alpha)", build_pipeline(DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=RANDOM_STATE)))
leaderboard()

**Lectura técnica**

- El árbol **captura no linealidades** que la logística no ve (hora × categoría × monto), sin necesidad de clip ni escalado.
- La brecha entre `train_pr_auc` y `pr_auc` en el leaderboard mide el **sobreajuste**. La poda reduce la varianza sin fijar la profundidad a mano.
- Un solo árbol sigue siendo **inestable**: cambia mucho con los datos. Eso motiva los ensembles.

<a name="6"></a>
## 6. Ensembles: voting y bagging

- **Bagging / Random Forest:** muchos árboles profundos sobre muestras bootstrap y subconjuntos de features; promediarlos **reduce la variance**.
- **Voting (soft):** promedia las probabilidades de modelos **distintos**; funciona si sus errores no están correlacionados.

In [ ]:
# =============================================================================
# 6.1 Random Forest and a soft-voting ensemble
# =============================================================================
rf = RandomForestClassifier(n_estimators=150, min_samples_leaf=5, max_features="sqrt", n_jobs=-1,
                            random_state=RANDOM_STATE)
cv_eval("random forest", build_pipeline(rf), n_jobs=1)

voting = VotingClassifier([
    ("logreg", clone(baseline_pipe.named_steps["model"])),
    ("tree", DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=RANDOM_STATE)),
    ("rf", clone(rf)),
], voting="soft")
cv_eval("soft voting (logreg + tree + rf)", build_pipeline(voting, clip_upper=baseline_params["prep__columns__num__clip__upper"]), n_jobs=1)
leaderboard()

**Lectura técnica:** el Random Forest supera al árbol solo con la misma información: la mejora viene de **promediar** árboles inestables. El voting no siempre gana: si un miembro es mucho peor que los otros, el promedio lo arrastra.

<a name="7"></a>
## 7. Boosting moderno: XGBoost, LightGBM y CatBoost

Boosting construye árboles **en secuencia**, cada uno corrigiendo los errores del anterior: **reduce el bias**. Las tres librerías implementan la misma idea con distinta forma de crecer los árboles:

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Crecimiento | por niveles (`max_depth`) | por hojas (`num_leaves`) | árboles simétricos (`depth`) |
| Desbalance | `scale_pos_weight` | `scale_pos_weight` | `auto_class_weights` |

Mismo pipeline (mismo one-hot): la comparación es justa. Los árboles no necesitan el escalado, pero tampoco les hace daño.

In [ ]:
# =============================================================================
# 7.1 The three libraries with comparable settings
# =============================================================================
boosters = {
    "xgboost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=SCALE_POS_WEIGHT, eval_metric="aucpr", n_jobs=-1, random_state=RANDOM_STATE),
    "lightgbm": LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, verbose=-1,
                               random_state=RANDOM_STATE),
    "catboost": CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, auto_class_weights="Balanced",
                                   verbose=0, thread_count=-1, random_seed=RANDOM_STATE),
}
for name, model in boosters.items():
    cv_eval(name, build_pipeline(model), n_jobs=1)
leaderboard()

**Lectura técnica**

- Los tres boosters superan con claridad a la logística y al Random Forest en PR-AUC.
- Las diferencias **entre** librerías suelen ser menores que la desviación entre folds: con parámetros por defecto no hay un ganador claro, y el tiempo de entrenamiento también cuenta.
- El **Brier** alto de los modelos con `scale_pos_weight` indica probabilidades infladas: ponderar la clase rara distorsiona $\hat p$ (sección 8).

<a name="8"></a>
## 8. Balanceo: ninguno, `class_weight` o SMOTE (hands-on 1)

Mismo modelo (LightGBM), mismos folds, tres estrategias. SMOTE va **dentro** del pipeline de `imblearn`, así que solo ve el train de cada fold.

In [ ]:
# =============================================================================
# 8.1 Same LightGBM, three imbalance strategies
# =============================================================================
lgbm_base = dict(n_estimators=300, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.8, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE)
smote_pipe = ImbPipeline([*build_preprocessor().steps, ("smote", SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)),
                          ("model", LGBMClassifier(**lgbm_base))]).set_output(transform="pandas")
strategies = {
    "lightgbm: none": build_pipeline(LGBMClassifier(**lgbm_base)),
    "lightgbm: class weight": build_pipeline(LGBMClassifier(**lgbm_base, scale_pos_weight=SCALE_POS_WEIGHT)),
    "lightgbm: SMOTE (in CV)": smote_pipe,
}
for name, estimator in strategies.items():
    cv_eval(name, estimator, n_jobs=1)
leaderboard().query("model.str.startswith('lightgbm')", engine="python")

**Lectura técnica**

- Con boosting, **no rebalancear** suele dar PR-AUC igual o mejor: el modelo ya ordena bien los casos, y el threshold se ajusta después.
- `class_weight` y SMOTE **inflan las probabilidades** (Brier peor): si se usan, hay que recalibrar o elegir el threshold por validación.
- Rebalancear es una **hipótesis a validar**, no una receta.

<a name="9"></a>
## 9. Optuna: CV como objetivo, early stopping y pruning (hands-on 2)

- **Objetivo:** PR-AUC media en los folds; la CV se fija **antes** de la búsqueda y es la misma en todos los trials.
- **TPE** propone hiperparámetros aprendiendo de los trials anteriores.
- **Early stopping:** dentro de cada fold se agregan árboles hasta que la validación deja de mejorar (elige `n_estimators`).
- **Pruning:** `MedianPruner` corta un trial si su score parcial queda bajo la mediana de los anteriores.

Para no repetir el preprocesamiento en cada trial, se ajusta **una vez por fold** (siempre solo con el train del fold).

In [ ]:
# =============================================================================
# 9.1 Precompute the preprocessed folds once (each fitted on its own training part)
# =============================================================================
hpo_cv = StratifiedKFold(3 if FAST_DEMO_MODE else N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_data = []
for tr, va in hpo_cv.split(X_train, y_train):
    prep = build_preprocessor().set_output(transform="pandas").fit(X_train.iloc[tr])
    fold_data.append((prep.transform(X_train.iloc[tr]), y_train.iloc[tr], prep.transform(X_train.iloc[va]), y_train.iloc[va]))


def suggest_lgbm(trial) -> dict:
    return {"learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0), "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True)}


def objective(trial) -> float:
    params, scores, best_iters = suggest_lgbm(trial), [], []
    for step, (Z_tr, y_tr, Z_va, y_va) in enumerate(fold_data):
        model = LGBMClassifier(n_estimators=1000, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE, **params)
        model.fit(Z_tr, y_tr, eval_set=[(Z_va, y_va)], eval_metric="average_precision",
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        scores.append(average_precision_score(y_va, model.predict_proba(Z_va)[:, 1]))
        best_iters.append(model.best_iteration_)
        trial.report(float(np.mean(scores)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("n_estimators", int(np.mean(best_iters)))
    return float(np.mean(scores))


t0 = time.perf_counter()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study.optimize(objective, n_trials=N_TRIALS)
optuna_seconds = time.perf_counter() - t0
pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
print(f"best PR-AUC {study.best_value:.4f} | pruned trials {pruned}/{N_TRIALS} | {optuna_seconds:.0f}s")
print("best params:", study.best_params, "| n_estimators:", study.best_trial.user_attrs["n_estimators"])

In [ ]:
# =============================================================================
# 9.2 Same budget with random search (no early stopping, fixed n_estimators)
# =============================================================================
t0 = time.perf_counter()
random_lgbm = RandomizedSearchCV(
    build_pipeline(LGBMClassifier(n_estimators=300, subsample_freq=1, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE)),
    {"model__learning_rate": loguniform(1e-2, 0.3), "model__num_leaves": randint(15, 256),
     "model__min_child_samples": randint(10, 201), "model__subsample": uniform(0.5, 0.5),
     "model__colsample_bytree": uniform(0.5, 0.5), "model__reg_lambda": loguniform(1e-3, 10)},
    n_iter=N_TRIALS, scoring="average_precision", cv=hpo_cv, n_jobs=1, random_state=RANDOM_STATE,
).fit(X_train, y_train)
random_seconds = time.perf_counter() - t0

complete = [t.value for t in study.trials if t.value is not None]
optuna_curve = pd.Series([t.value if t.value is not None else np.nan for t in study.trials]).cummax()
random_curve = pd.Series(random_lgbm.cv_results_["mean_test_score"]).cummax()
ax = pd.DataFrame({"Optuna (TPE + pruning)": optuna_curve, "random search": random_curve}).plot(marker="o", figsize=(8, 4.5))
ax.set(title="Mejor PR-AUC acumulada por trial (mismo presupuesto)", xlabel="trial", ylabel="mejor PR-AUC")
plt.show()
pd.DataFrame([{"search": "optuna", "best_pr_auc": study.best_value, "seconds": optuna_seconds, "pruned": pruned},
              {"search": "random", "best_pr_auc": random_lgbm.best_score_, "seconds": random_seconds, "pruned": 0}])

**Lectura técnica**

- Optuna gasta el presupuesto en zonas prometedoras y corta trials malos: suele llegar antes a un buen valor y en menos tiempo.
- El early stopping usa el fold de validación para elegir `n_estimators`, así que el score de Optuna queda **algo optimista**. La evaluación final sigue siendo en test.
- Si la diferencia entre búsquedas es menor que la dispersión entre folds, lo honesto es decir que **empatan**.

<a name="10"></a>
## 10. Modelo final, evaluación única en test e inferencia

Se reentrena el mejor LightGBM **dentro del mismo pipeline**, se elige el threshold por costo con CV (como en el notebook 01) y se abre el test una vez. Luego se compara con el baseline en los **datos futuros** (`fraudTest.csv`).

In [ ]:
# =============================================================================
# 10.1 Final model: tuned LightGBM, cost threshold, one evaluation on test
# =============================================================================
final_lgbm = LGBMClassifier(n_estimators=study.best_trial.user_attrs["n_estimators"], subsample_freq=1, n_jobs=-1,
                            verbose=-1, random_state=RANDOM_STATE,
                            **{k: v for k, v in study.best_params.items()})
cv_eval("lightgbm tuned (optuna)", build_pipeline(final_lgbm), n_jobs=1)

cost_scorer = make_scorer(lambda y_true, y_pred: -expected_cost(y_true, y_pred))
tuned = {name: TunedThresholdClassifierCV(pipe, scoring=cost_scorer, cv=skf, n_jobs=-1).fit(X_train, y_train)
         for name, pipe in {"logreg (baseline 01)": baseline_pipe, "lightgbm tuned (optuna)": build_pipeline(final_lgbm)}.items()}
test_table = pd.DataFrame([binary_report(y_test, m.estimator_.predict_proba(X_test)[:, 1], m.best_threshold_, name)
                           for name, m in tuned.items()])
display(leaderboard())
test_table

In [ ]:
# =============================================================================
# 10.2 Save v2 and compare both models on six months of future transactions
# =============================================================================
final_model = tuned["lightgbm tuned (optuna)"].estimator_
final_threshold = float(tuned["lightgbm tuned (optuna)"].best_threshold_)
joblib.dump(final_model, ARTIFACT_DIR / "fraud_pipeline_v2.joblib")
(ARTIFACT_DIR / "model_card_v2.json").write_text(json.dumps({
    "model_name": "fraud_pipeline_v2", "model": "LGBMClassifier (Optuna)", "threshold": final_threshold,
    "params": {**study.best_params, "n_estimators": study.best_trial.user_attrs["n_estimators"]},
    "input_schema": INPUT_SCHEMA, "features_code": "notebook 01 cell 7.1 / notebook 02 cell 4.3 (SHARED PREP BLOCK 2/3)",
    "test_metrics": {k: float(v) for k, v in test_table.iloc[1].items() if isinstance(v, (int, float, np.floating))},
}, indent=2))

production = load_fraud_csv(DATA_DIR / FILES["production"])
X_future, y_future = validate_schema(production), production[TARGET]
future_table = pd.DataFrame([binary_report(y_future, m.estimator_.predict_proba(X_future)[:, 1], m.best_threshold_, name)
                             for name, m in tuned.items()])
print(f"saved {ARTIFACT_DIR / 'fraud_pipeline_v2.joblib'} | future rows: {len(production):,}")
future_table

In [ ]:
# =============================================================================
# 10.3 What the booster learned: top features by gain
# =============================================================================
booster = final_model.named_steps["model"]
importance = pd.Series(booster.booster_.feature_importance(importance_type="gain"),
                       index=final_model.named_steps["prep"].get_feature_names_out()).sort_values(ascending=False)
ax = (importance.head(12) / importance.sum()).sort_values().plot.barh(figsize=(8, 5))
ax.set(title="LightGBM: importancia por ganancia (top 12)", xlabel="fracción de la ganancia total")
plt.show()

**Lectura técnica**

- En el test interno y en los datos futuros, el boosting reduce el **costo** frente a la logística del notebook 01, con el mismo pipeline de features.
- Las variables más importantes coinciden con el EDA del notebook 01: **monto**, **horas nocturnas** y **categoría**.
- La caída de desempeño en datos futuros afecta a los dos modelos: el optimismo del split aleatorio no lo resuelve un modelo más potente.

<a name="11"></a>
## 11. Ejercicios y cierre

1. **CatBoost con categóricas nativas:** en lugar del one-hot, pase `category`, `state`, `hour` como `cat_features`. ¿Mejora frente al pipeline común?
2. **Recalibrar:** aplique `CalibratedClassifierCV` al LightGBM con `class_weight`. ¿Se recupera el Brier? ¿Cambia el threshold por costo?
3. **Más presupuesto:** pase `FAST_DEMO_MODE = False`. ¿La diferencia entre Optuna y random search se mantiene?
4. **Validación temporal:** repita la sección 9 con `TimeSeriesSplit` sobre la historia ordenada por fecha. ¿El score se acerca al de `fraudTest`?

**La frase que debe sobrevivir:** un modelo más potente solo gana si lo hace **con la misma CV, la misma métrica y el mismo pipeline**, y si sostiene la mejora en datos que nunca vio.

**Sesión 2:** la teoría detrás de XGBoost, LightGBM y CatBoost (objetivo regularizado, histogramas, *ordered boosting*) sobre este mismo caso: `ml_boosting_optuna.ipynb`.